In [0]:
"""
ETL PIPELINE PRACTICE SCENARIOS
REAL-TIME NOTEBOOK EXERCISES


# =========================================================
# PROJECT 1 - RETAIL SALES ETL PIPELINE
# =========================================================


DATASETS:
1. sales.csv
2. customers.json
3. products.parquet
4. stores.csv


# -----------------------------
# PHASE 1 - INGESTION
# -----------------------------

1. Read all datasets from CSV, JSON, and Parquet.
2. Print schema of all DataFrames.
3. Verify row count of each dataset.
4. Check for duplicate records.

# -----------------------------
# PHASE 2 - CLEANING
# -----------------------------

5. Convert sales_date to DateType.
6. Remove rows where sales_amount is null.
7. Replace null city values with 'Unknown'.
8. Remove duplicate sales records.
9. Rename product_name column to item_name.
10. Drop unwanted columns.

# -----------------------------
# PHASE 3 - TRANSFORMATIONS
# -----------------------------

11. Add a new column tax_amount = 18% of sales_amount.
12. Add total_amount = sales_amount + tax_amount.
13. Create sales_category:
      > 5000 → High
      2000-5000 → Medium
      < 2000 → Low

14. Add order_year from sales_date.
15. Add order_month from sales_date.

# -----------------------------
# PHASE 4 - JOINS
# -----------------------------

16. Join sales with customers.
17. Join sales with products.
18. Join sales with stores.
19. Show customer_name, city, item_name, sales_amount.
20. Find sales records with missing product details using ANTI JOIN.

# -----------------------------
# PHASE 5 - BUSINESS ANALYSIS
# -----------------------------

21. Find total sales per city.
22. Find top 5 customers by purchase amount.
23. Find most sold product.
24. Find average sales per store.
25. Find total sales by product category.
26. Find monthly sales trend.
27. Find stores with sales greater than 50K.

# -----------------------------
# PHASE 6 - ADVANCED ETL
# -----------------------------

28. Create a window function to rank customers by spending.
29. Find highest sale in each city.
30. Find cumulative sales per month.
31. Identify repeat customers.

# -----------------------------
# PHASE 7 - LOAD / SAVE
# -----------------------------

32. Save cleaned data as Parquet.
33. Save business summary as CSV.
34. Partition final output by city.
35. Read saved Parquet back and verify schema.
36. Append new sales data using unionByName().

# =========================================================
# PROJECT 2 - BANKING TRANSACTION PIPELINE
# =========================================================


DATASETS:
1. transactions.csv
2. customers.json
3. branches.parquet


1. Read and validate all datasets.
2. Convert transaction_date to DateType.
3. Filter failed transactions.
4. Identify duplicate transaction IDs.
5. Calculate total transaction amount per customer.
6. Find suspicious transactions greater than 1 lakh.
7. Join customers with transactions.
8. Find branches with highest transaction volume.
9. Calculate daily transaction totals.
10. Identify inactive customers.
11. Use window functions to rank top customers.
12. Save final fraud analysis report.

# =========================================================
# PROJECT 3 - LOG PROCESSING PIPELINE
# =========================================================


DATASETS:
1. application_logs.json
2. server_details.csv


1. Read log files.
2. Extract log_date and log_hour.
3. Filter ERROR logs.
4. Count errors by server.
5. Find most frequent error message.
6. Join logs with server details.
7. Calculate hourly error trend.
8. Save error summary report.

# =========================================================
# PROJECT 4 - HEALTHCARE DATA PIPELINE
# =========================================================

DATASETS:
1. patients.csv
2. appointments.json
3. doctors.parquet

1. Clean patient records.
2. Remove invalid ages.
3. Convert appointment_date to DateType.
4. Join patients with appointments.
5. Find patients with multiple appointments.
6. Find top doctors by appointment count.
7. Calculate average patient age by city.
8. Save final healthcare analytics report.

"""

In [0]:
df_products = spark.read.table("dataframeassigment.etl.products")
df_products.write.mode("overwrite").saveAsTable("dataframeassigment.etl.products_parquet")

In [0]:
# PHASE 1 - INGESTION
# 1. Read all datasets from CSV, JSON, and Parquet.
df_customers = spark.read.table("dataframeassigment.etl.customers")
df_products = spark.read.table("dataframeassigment.etl.products")
df_sales = spark.read.table("dataframeassigment.etl.sales")
df_stores = spark.read.table("dataframeassigment.etl.stores")

In [0]:
# PHASE 1 - INGESTION
# 2. Print schema of all DataFrames.
print("Schema of customers: ")
df_customers.printSchema()
print("Schema of products: ")
df_products.printSchema()
print("Schema of sales: ")
df_sales.printSchema()
print("Schema of stores: ")
df_stores.printSchema()

In [0]:
# PHASE 1 - INGESTION
# 3. Verify row count of each dataset.
print("Number of rows in customers: ", df_customers.count())
print("Number of rows in products: ", df_products.count())
print("Number of rows in sales: ", df_sales.count())
print("Number of rows in stores: ", df_stores.count())


In [0]:
# PHASE 1 - INGESTION
# 4. Check for duplicate records.

print("Duplicate records of sales: ")
df_sales.groupBy("sale_id").count().filter("count > 1").display()

print("Duplicate records of customers: ")
df_customers.groupBy("customer_id").count().filter("count > 1").display()

print("Dulplicate records of products: ")
df_products.groupBy("product_id").count().filter("count > 1").display()

print("Duplicate records of stores: ")
df_stores.groupBy("store_id").count().filter("count>1").display()

In [0]:
# PHASE 2 - CLEANING
# 5. Convert sales_date to DateType.

from pyspark.sql.functions import to_date, col

df_sales = df_sales.withColumn("sales_date", to_date(col("sales_date"), "yyyy-MM-dd"))
df_sales.show()

In [0]:
# PHASE 2 - CLEANING
# 6. Remove rows where sales_amount is null.
from pyspark.sql.functions import col
df_sales.filter(col("sales_amount").isNotNull()).show()



In [0]:
# PHASE 2 - CLEANING
# 7. Replace null city values with 'Unknown'.

from pyspark.sql.functions import col
df_customers = df_customers.fillna({"city": "Unknown"})
df_customers.display()

In [0]:
# PHASE 2 - CLEANING
# 8. Remove duplicate sales records.
df_sales = df_sales.dropDuplicates()
print(f"Number of rows after removing duplicates: {df_sales.count()}")

In [0]:
# PHASE 2 - CLEANING
# 9. Rename product_name column to item_name.
df_products = df_products.withColumnRenamed("product_name", "item_name")
df_products.show()


In [0]:
# PHASE 2 - CLEANING
# 10. Drop unwanted columns.
df_customers.drop("age").display()

In [0]:
# PHASE 3 - TRANSFORMATIONS
# 11. Add a new column tax_amount = 18% of sales_amount.
from pyspark.sql.functions import col
df_sales = df_sales.withColumn("tax_amount", col("sales_amount")*18/100)
df_sales.display()

In [0]:
# PHASE 3 - TRANSFORMATIONS
# 12. Add total_amount = sales_amount + tax_amount.

from pyspark.sql.functions import col
df_sales.withColumn("total_amount", col("sales_amount")+col("tax_amount")).display()

In [0]:
# PHASE 3 - TRANSFORMATIONS
# 13. Create sales_category:
#       > 5000 → High
#       2000-5000 → Medium
#       < 2000 → Low

from pyspark.sql.functions import col, when
df_sales = df_sales.withColumn("sales_category", when(col("sales_amount")>5000, "High").when(col("sales_amount")>=2000, "Medium").otherwise("Low"))
df_sales.display()

In [0]:
# PHASE 3 - TRANSFORMATIONS
# 14. Add order_year from sales_date.
from pyspark.sql.functions import col, year
df_sales = df_sales.withColumn("order_year", year(col("sales_date")))
df_sales.display()

In [0]:
# PHASE 3 - TRANSFORMATIONS
# 15. Add order_month from sales_date.
from pyspark.sql.functions import col, month
df_sales = df_sales.withColumn("order_month", month(col("sales_date")))
df_sales.display()

In [0]:
# PHASE 4 - JOINS
# 16. Join sales with customers.
df_sales_customers = df_sales.join(df_customers, "customer_id", "inner")
df_sales_customers.display()

In [0]:
# PHASE 4 - JOINS
# 17. Join sales with products.
df_sales_products = df_sales_customers.join(df_products, "product_id", "inner")
df_sales_products.display()

In [0]:
# PHASE 4 - JOINS
# 18. Join sales with stores.
df_final = df_sales_products.join(df_stores, "store_id", "inner")
df_final.display()

In [0]:
# PHASE 4 - JOINS
# 19. Show customer_name, city, item_name, sales_amount.
df_final.select("customer_id", "first_name", df_sales_customers["city"], "item_name", "sales_amount").display()

In [0]:
# PHASE 4 - JOINS
# 20. Find sales records with missing product details using ANTI JOIN.
df_sales.join(df_products, "product_id", "left_anti").display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 21. Find total sales per city.
from pyspark.sql.functions import sum
df_final.groupBy(df_sales_customers["city"]).agg(sum("sales_amount").alias("total_sales")).display()

In [0]:
df_final.display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 22. Find top 5 customers by purchase amount.
from pyspark.sql.functions import sum, col
df_final.groupBy("category", "first_name").sum("sales_amount").orderBy(col("sum(sales_amount)").desc()).limit(5).display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 23. Find most sold product.
df_final.groupBy("item_name").sum("quantity").orderBy(col("sum(quantity)").desc()).display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 24. Find average sales per store.
df_final.groupBy("store_name").avg("sales_amount").display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 25. Find total sales by product category.
df_final.groupBy("category").sum("sales_amount").orderBy(col("sum(sales_amount)")).display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 26. Find monthly sales trend.
df_final.groupBy("order_month").sum("sales_amount").orderBy("order_month").display()

In [0]:
# PHASE 5 - BUSINESS ANALYSIS
# 27. Find stores with sales greater than 1 lakh.
df_final.groupBy("store_name").sum("sales_amount").filter(col("sum(sales_amount)") > 50000).display()

In [0]:
# PHASE 6 - ADVANCED ETL
# 28. Create a window function to rank customers by spending.

from pyspark.sql.functions import sum, col, rank
from pyspark.sql.window import Window

df_customer_spending = df_final.groupBy("customer_id","first_name").agg(sum("sales_amount").alias("total_spending"))

window_spec = Window.orderBy(col("total_spending").desc())

df_customer_spending.withColumn("rank",rank().over(window_spec)).display()

In [0]:
# PHASE 6 - ADVANCED ETL
# 29. Find highest sale in each city.

from pyspark.sql.functions import max
df_final.groupBy(df_sales_customers["city"]).agg(max("sales_amount").alias("max_sale")).display()

In [0]:
# PHASE 6 - ADVANCED ETL
# 30. Find cumulative sales per month.

from pyspark.sql.functions import month, sum
from pyspark.sql.window import Window

df_monthly = df_final.groupBy(month("sales_date").alias("month")).agg(sum("sales_amount").alias("monthly_sales"))

window_spec = Window.orderBy("month")

df_monthly.withColumn("cumulative_sales",sum("monthly_sales").over(window_spec)).display()

In [0]:
# PHASE 6 - ADVANCED ETL
# 31. Identify repeat customers.

from pyspark.sql.functions import count

df_final.groupBy("customer_id","first_name").agg(count("sale_id").alias("total_orders")).filter(col("total_orders") > 1).display()

In [0]:
# PHASE 7 - LOAD / SAVE
# 32. Save cleaned data as Parquet.
df_final_clean = df_final.drop(df_stores["city"]).drop(df_stores["state"])
df_final_clean.write.mode("overwrite").saveAsTable("dataframeassigment.etl.final_parquet")


In [0]:
# PHASE 7 - LOAD / SAVE
# 33. Save business summary as CSV.
df_final_clean.write.mode("overwrite").saveAsTable("dataframeassigment.etl.final_csv")

In [0]:
# PHASE 7 - LOAD / SAVE
# 34. Partition final output by city.

df_final_clean.write.mode("overwrite").option("overwriteSchema", "true").partitionBy("city").saveAsTable("dataframeassigment.etl.partitioned_report")